In [4]:
import os
from pyspark.sql import SparkSession

# 🚀 STEP 1: Set identity for HDFS and directory for datasets.txt
os.environ['HADOOP_USER_NAME'] = 'root'
if os.path.exists('/home/jovyan/work'):
    os.chdir('/home/jovyan/work')

print("🧹 Cleaning up stale Spark sessions...")
try:
    SparkSession.builder.getOrCreate().stop()
except:
    pass

print("🚀 Initializing Spark with HDFS Patches & MongoDB Connector...")
# Note: Ensure you have internet access to download the mongo-spark-connector jar
spark = SparkSession.builder \
    .appName("Ecommerce_Dual_Ingestion_Final") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.2") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.caseSensitive", "true") \
    .config("spark.hadoop.dfs.client.use.datanode.hostname", "true") \
    .config("spark.hadoop.dfs.datanode.use.datanode.hostname", "true") \
    .getOrCreate()

print(f"✅ Infrastructure Ready! Current Working Directory: {os.getcwd()}")

🧹 Cleaning up stale Spark sessions...
🚀 Initializing Spark with HDFS Patches & MongoDB Connector...
✅ Infrastructure Ready! Current Working Directory: /home/jovyan/work


In [5]:
# ==========================================
# PHASE 1 & 2: HDFS (Meta) + MongoDB (Review)
# ==========================================
config_file = "datasets.txt"

if not os.path.exists(config_file):
    print(f"❌ Error: {config_file} not found in {os.getcwd()}! Please check file location.")
else:
    with open(config_file, 'r') as f:
        lines = [l.strip() for l in f if l.strip()]
    
    print(f"📋 Found {len(lines)} datasets to process.\n")

    for line in lines:
        dataset_name, table_name = [i.strip() for i in line.split(',')]
        print(f"{'='*60}\n🔄 Current Task: [{dataset_name}]\n{'='*60}")

        # Path Definitions
        meta_local_in = f"file://{os.path.abspath(f'data/meta/meta_{dataset_name}.jsonl')}"
        rev_local_in = f"file://{os.path.abspath(f'data/review/{dataset_name}.jsonl')}"
        
        hdfs_dest = f"hdfs://namenode:9000/ecommerce/data/meta_{table_name}.parquet"
        mongo_uri = f"mongodb://mongodb:27017/amazon_reviews.{table_name}"

        # ---------------------------------------------------------
        # TASK A: Metadata -> HDFS
        # ---------------------------------------------------------
        try:
            print(f"📦 [META] Reading from local: {meta_local_in}")
            df_meta = spark.read.option("mode", "DROPMALFORMED").json(meta_local_in)
            
            print(f"💾 [META] Uploading to HDFS: {hdfs_dest}")
            df_meta.write.mode("overwrite").parquet(hdfs_dest)
            print(f"✅ [META] Success: {table_name} stored in HDFS.")
        except Exception as e:
            print(f"❌ [META] Failed: {str(e)[:200]}")

        # ---------------------------------------------------------
        # TASK B: Reviews -> MongoDB
        # ---------------------------------------------------------
        try:
            print(f"📦 [REV] Reading from local: {rev_local_in}")
            df_rev = spark.read.option("mode", "DROPMALFORMED").json(rev_local_in)
            
            print(f"💾 [REV] Saving to MongoDB Collection: {table_name}")
            df_rev.write.format("mongo").option("uri", mongo_uri).mode("overwrite").save()
            print(f"✅ [REV] Success: {table_name} reviews stored in Mongo.")
        except Exception as e:
            print(f"❌ [REV] Failed: {str(e)[:200]}")

    print("\n✨ ALL TASKS FINISHED! DATA LAKE IS READY.")

📋 Found 2 datasets to process.

🔄 Current Task: [Electronics]
📦 [META] Reading from local: file:///home/jovyan/work/data/meta/meta_Electronics.jsonl
💾 [META] Uploading to HDFS: hdfs://namenode:9000/ecommerce/data/meta_elec.parquet
✅ [META] Success: elec stored in HDFS.
📦 [REV] Reading from local: file:///home/jovyan/work/data/review/Electronics.jsonl
💾 [REV] Saving to MongoDB Collection: elec
✅ [REV] Success: elec reviews stored in Mongo.
🔄 Current Task: [Cell_Phones_and_Accessories]
📦 [META] Reading from local: file:///home/jovyan/work/data/meta/meta_Cell_Phones_and_Accessories.jsonl
💾 [META] Uploading to HDFS: hdfs://namenode:9000/ecommerce/data/meta_cellp.parquet
❌ [META] Failed: An error occurred while calling o179.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 6 in stage 5.0 failed 1 times, most recent failure: Lost task 6.0 in stage 5.0 (
📦 [REV] Reading from local: file:///home/jovyan/work/data/review/Cell_Phones_and_Accessories.jsonl
💾 [REV] 

In [6]:
# ==========================================
# PHASE 2.5: MongoDB Storage Verification
# ==========================================
import os

config_file = "datasets.txt"

if not os.path.exists(config_file):
    print(f"❌ Error: {config_file} not found!")
else:
    with open(config_file, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]
    
    print(f"🔎 Checking MongoDB 'amazon_reviews' database...\n")

    for line in lines:
        _, table_name = [item.strip() for item in line.split(',')]
        mongo_uri = f"mongodb://mongodb:27017/amazon_reviews.{table_name}"
        
        try:
            # 🚀 Loading data directly from MongoDB
            df_mongo = spark.read \
                .format("mongo") \
                .option("uri", mongo_uri) \
                .load()
            
            # Get the total count
            total_count = df_mongo.count()
            
            print(f"{'='*50}")
            print(f"📦 Collection: [{table_name}]")
            print(f"📈 Total Documents: {total_count}")
            print(f"{'='*50}")
            
            # Show a small preview of the data
            if total_count > 0:
                print(f"👀 Data Sample (First 5 reviews):")
                df_mongo.select("reviewerID", "asin", "overall", "reviewText") \
                        .show(5, truncate=50)
            else:
                print("⚠️ This collection is empty!")
                
        except Exception as e:
            print(f"❌ Could not read collection [{table_name}]: {str(e)[:100]}")

    print("\n✨ Verification finished!")

🔎 Checking MongoDB 'amazon_reviews' database...



ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


❌ Could not read collection [elec]: An error occurred while calling o196.count
❌ Could not read collection [cellp]: [Errno 111] Connection refused

✨ Verification finished!


In [7]:
# ==========================================
# PHASE 2.6: Dual Storage Verification (HDFS & Local)
# ==========================================
import os

config_file = "datasets.txt"
local_lake = "data_lake"

# 🚀 Initialize HDFS FileSystem via JVM
gateway = spark.sparkContext._gateway
jvm = gateway.jvm
hadoop_conf = spark._jsc.hadoopConfiguration()
fs_hdfs = jvm.org.apache.hadoop.fs.FileSystem.get(
    jvm.java.net.URI("hdfs://namenode:9000"), 
    hadoop_conf
)

if not os.path.exists(config_file):
    print(f"❌ Error: {config_file} not found!")
else:
    with open(config_file, 'r') as f:
        datasets = [line.strip().split(',') for line in f if line.strip()]

    print(f"🔎 Starting Cross-Platform Storage Audit...\n")

    for ds_name, table_name in datasets:
        ds_name, table_name = ds_name.strip(), table_name.strip()
        print(f"{'='*60}")
        print(f"📊 Dataset: [{ds_name}]")
        print(f"{'='*60}")

        # --- 1. Check HDFS (Port 9000) ---
        hdfs_path = f"/ecommerce/data/meta_{table_name}.parquet"
        try:
            hdfs_exists = fs_hdfs.exists(jvm.org.apache.hadoop.fs.Path(hdfs_path))
            if hdfs_exists:
                # Use Spark to get row count from HDFS
                count = spark.read.parquet(f"hdfs://namenode:9000{hdfs_path}").count()
                print(f"🌐 [HDFS] Status: FOUND | Records: {count}")
                print(f"   ∟ Path: hdfs://namenode:9000{hdfs_path}")
            else:
                print(f"🌐 [HDFS] Status: NOT FOUND")
        except Exception as e:
            print(f"🌐 [HDFS] Status: ERROR (Connection issue)")

        # --- 2. Check Local (Windows/Docker Work Folder) ---
        local_path = os.path.join(local_lake, f"meta_{table_name}.parquet")
        if os.path.exists(local_path):
            try:
                # Use Spark to get row count from Local
                count = spark.read.parquet(f"file://{os.path.abspath(local_path)}").count()
                print(f"💻 [LOCAL] Status: FOUND | Records: {count}")
                print(f"   ∟ Path: {local_path}")
            except Exception as e:
                print(f"💻 [LOCAL] Status: FILE EXISTS BUT UNREADABLE")
        else:
            print(f"💻 [LOCAL] Status: NOT FOUND")

    print(f"\n{'='*60}")
    print("✨ Audit Complete! You can now decide which source to use for Phase 3.")

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


ConnectionRefusedError: [Errno 111] Connection refused

In [8]:
# ==========================================
# PHASE 2.7: Lightweight Mongo Check (No Spark)
# ==========================================
try:
    from pymongo import MongoClient
    # Use the internal docker network name 'mongodb'
    client = MongoClient('mongodb://mongodb:27017/', serverSelectionTimeoutMS=2000)
    db = client['amazon_reviews']
    
    print("🔎 Scanning MongoDB collections...")
    collections = db.list_collection_names()
    
    if not collections:
        print("⚠️ No collections found in 'amazon_reviews'.")
    else:
        for col_name in collections:
            count = db[col_name].estimated_document_count()
            print(f"✅ Found [{col_name}]: {count} documents")
            
except Exception as e:
    print(f"❌ Connection Failed: {e}")
    print("💡 Tip: Make sure your MongoDB container is running (docker ps).")

🔎 Scanning MongoDB collections...
✅ Found [elec]: 43886944 documents
✅ Found [cellp]: 20812945 documents
